# 1. Train Scale Experiments

This notebook bootstraps the clean next-scale repo in Colab, runs the required first-run experiment, and lets you optionally launch the broader sweep after that succeeds.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = 'AttnResGPT-next-scale'
REPO_URL = os.environ.get('ATTNRES_REPO_URL', 'https://github.com/your-org/AttnResGPT-next-scale.git')
REPO_DIR = Path('/content') / REPO_NAME
DRIVE_REPO_DIR = Path('/content/drive/MyDrive') / REPO_NAME

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as error:
    print(f'Drive mount skipped: {error}')

if not REPO_DIR.exists():
    if DRIVE_REPO_DIR.exists():
        subprocess.run(['cp', '-r', str(DRIVE_REPO_DIR), str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Using repo at: {REPO_DIR}')


In [ ]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())


## Required First-Run Experiment

This runs SMALL only, contexts 128 and 512, baseline plus AttnRes, 300 steps each.

In [ ]:
!python experiments/scale_experiment.py --config configs/first_run.yaml --skip-existing

In [ ]:
import pandas as pd
from pathlib import Path

summary_path = Path('outputs/logs/run_summaries.csv')
paired_path = Path('outputs/logs/paired_comparisons.csv')

summary_df = pd.read_csv(summary_path)
paired_df = pd.read_csv(paired_path)

display(summary_df[['model', 'size', 'context', 'val_loss', 'perplexity', 'second_half_loss', 'mean_activation_norm_last_layer', 'mean_early_contribution', 'mean_late_contribution']])
display(paired_df)


## Optional Full Sweep

Only run these after the first-run outputs look healthy.

In [ ]:
# !python experiments/scale_experiment.py --config configs/small.yaml --skip-existing
# !python experiments/scale_experiment.py --config configs/medium.yaml --skip-existing